# V57 — 25-Action Directional Spec

A/B test against V56a. Everything identical **except the action space and environment**.

## Action space (25 dims, all SAC-native Box[-1,1])

| dims | range | meaning |
|------|-------|---------|
| `action[0:MAX_RUNNERS]` | [-1, 1] | Per-runner signal. >0 = back, <0 = lay, magnitude = confidence |
| `action[MAX_RUNNERS]` | [-1, 1] | Stake fraction → (x+1)/2 maps to [0,1] of available capital |

At each bar the agent chooses how much of its available bankroll to deploy (`stake_frac`)
and splits it proportionally to `|signal[r]|` across all runners.  Runners with signal ≈ 0
receive near-zero allocation naturally — no threshold gate needed.

**Key advantage over V56a's 24-dim offset spec:** the do-nothing policy is
a *single-dimension* discovery — push `action[MAX_RUNNERS]` to -1 (stake→0).
V56a required all 24 dims below -0.5 simultaneously (24-dimensional coordination).

## Execution model

| direction | fill price |
|-----------|-----------|
| back  (signal > 0) | `bl` — cross the spread, take the best lay offer |
| lay   (signal < 0) | `bb` — cross the spread, take the best back offer |

No passive orders, no queue simulation.  All fills are market orders.

## Observation (422 dims)

Identical static runner features to V56a (26 dims/runner).
Own-order features replaced by 6 position-tracking dims:

| dim | meaning |
|-----|---------|
| `pos_back` | back stake / MAX_RUNNER_EXP |
| `pos_lay`  | lay liability / MAX_RUNNER_EXP |
| `bpx_dist` | avg back price vs current bb (normalized) |
| `lpx_dist` | avg lay price vs current bb (normalized) |
| `green_r`  | per-runner green value (normalized) |
| `pnl_r`    | same as green_r (placeholder for net PnL) |

Market features: V56a's 24 + 2 new (total_risk_frac, available_frac) = 26.

`OBS_DIM = 12×32 + 26 + 12 = 422`

## Stage A gate

Same as V56a FastTrack:
- random ~ -$200
- do-nothing = $0
- trained must reach ≥ -$1 to pass


## Cell 0 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 0b — Environment check

In [ ]:
import sys, platform, subprocess
print(f"platform  {platform.system()} {platform.machine()}")
assert platform.system() == "Linux", (
    "Not running on Colab. The WinError 1114 you saw came from the local Anaconda "
    "kernel, not from here. Check the runtime selector top-right.")

try:
    import torch
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q","torch"],check=True)
    import torch
try:
    import stable_baselines3
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q",
                    "stable-baselines3[extra]","gymnasium"],check=True)
    import stable_baselines3

print(f"torch     {torch.__version__}  cuda={torch.cuda.is_available()}")
print(f"sb3       {stable_baselines3.__version__}")
if torch.cuda.is_available():
    print(f"gpu       {torch.cuda.get_device_name(0)}")
else:
    print("gpu       none — Runtime > Change runtime type > T4 GPU. "
          "V52 ran at 3 fps on CPU; that was the bottleneck.")

## Cell 1 — Config

In [ ]:
import os, sys, json, gzip, zlib, time, math, warnings, datetime as dt
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ── Drive layout (identical to V56a) ─────────────────────────────────────────
DRIVE_DATA = None
if DRIVE_DATA is None:
    root = Path("/content/drive/MyDrive")
    hits = list(root.rglob("recordings/*.ndjson.gz"))
    if not hits:
        raise FileNotFoundError(
            f"No */recordings/*.ndjson.gz under {root}.\n"
            "Set DRIVE_DATA manually.")
    parents = [p.parent.parent for p in hits]
    DRIVE_DATA = max(set(parents), key=parents.count)

print(f"drive data : {DRIVE_DATA}")
n_drive = len(list((DRIVE_DATA / "recordings").glob("*.ndjson.gz")))
assert n_drive > 0
print(f"on drive   : {n_drive} recordings")

DATA_DIR = Path("/content/bf_local"); DATA_DIR.mkdir(exist_ok=True)
REC_DIR  = DATA_DIR / "recordings";   REC_DIR.mkdir(exist_ok=True)
have = {f.name for f in REC_DIR.glob("*.ndjson.gz")}
todo = [f for f in (DRIVE_DATA/"recordings").glob("*.ndjson.gz") if f.name not in have]
if todo:
    import shutil, time as _t; _t0=_t.time()
    for f in todo: shutil.copy2(f, REC_DIR/f.name)
    print(f"copied     : {len(todo)} new in {_t.time()-_t0:.0f}s")
CAT_DIR = DATA_DIR/"catalogues"; CAT_DIR.mkdir(exist_ok=True)
_hc = {f.name for f in CAT_DIR.glob("*.json")}
_tc = [f for f in (DRIVE_DATA/"catalogues").glob("*.json") if f.name not in _hc]
if _tc:
    import shutil as _sh
    for f in _tc: _sh.copy2(f, CAT_DIR/f.name)
    print(f"catalogues : {len(_tc)} copied")
n_local = len(list(REC_DIR.glob("*.ndjson.gz")))
print(f"local      : {n_local} recordings")
assert n_local == n_drive

CACHE_DIR = DRIVE_DATA / "rl_cache"; CACHE_DIR.mkdir(exist_ok=True)
CORPUS_FREEZE = None
ARM     = "V57_action_spec"
RUN_DIR = DRIVE_DATA / f"rl_runs_{ARM}"; RUN_DIR.mkdir(exist_ok=True)

# ── Episode / bars ───────────────────────────────────────────────────────────
BAR_S          = 1.0
MAX_RUNNERS    = 12
MIN_PROB       = 0.01
PREOFF_START_S = 600
K_OFFSETS      = 4      # kept for cache-key compatibility with V56a CACHE_SCHEMA 6

# ── New action-spec parameters ───────────────────────────────────────────────
STARTING_BANK  = 1000.0   # initial bankroll ($)
MIN_BET        = 1.0      # minimum allocation per runner ($)
MAX_RUNNER_EXP = 200.0    # normalisation constant for position sizes

# ── Economics ────────────────────────────────────────────────────────────────
COMMISSION = 0.08         # default if catalogue missing (8%)

# ── SAC / entropy ────────────────────────────────────────────────────────────
GAMMA           = 0.99
# RAISED vs V56a (was effectively ~0.001 — caused entropy collapse by ep 50)
ENT_COEF_FLOOR  = 0.05
ENT_COEF_CEIL   = 2.0
# 25 action dims; fully-uniform H ≈ 25*ln(2) ≈ +17.3 nats.
# Target -8 is moderately low — policy can specialize but can't collapse.
TARGET_ENTROPY  = -8.0

# ── Training ─────────────────────────────────────────────────────────────────
SEED            = 42
FASTTRACK_SEED  = SEED        # alias for verbatim V56a build_cache cell
FILL_MODE       = "touch"     # market orders only
FASTTRACK_N     = 5
PASSES_PER_RACE = 1000

print(f"  ARM            : {ARM}")
print(f"  MAX_RUNNERS    : {MAX_RUNNERS}")
print(f"  STARTING_BANK  : ${STARTING_BANK:.0f}")
print(f"  COMMISSION     : {COMMISSION:.0%}")
print(f"  ENT_COEF_FLOOR : {ENT_COEF_FLOOR}")
print(f"  TARGET_ENTROPY : {TARGET_ENTROPY}")
print(f"  action dims    : {MAX_RUNNERS + 1} (24 runner signals + 1 stake fraction)")
print("Config OK")


## Cell 1b — Catalogue audit

In [ ]:
### Cell 1b — CATALOGUE AUDIT (run before anything else) ###
# The Kilmore catalogue showed marketBaseRate = 8.0, i.e. commission is 8%, not the 5%
# hardcoded through V53/Option 1/V54-56. It also showed priceLadderDescription = CLASSIC.
# If ANY market is FINEST (0.01 increments throughout), to_ticks() is silently wrong for
# it and every tick-space number in this project is invalid for those races.
from collections import Counter
import json as _json

lad=Counter(); rate=Counter(); rt=Counter(); miss=0
_AUDIT_DIR = DRIVE_DATA/"catalogues"
for p in sorted(_AUDIT_DIR.glob("*.json")):
    try:
        d=_json.loads(p.read_text())["catalogue"]; desc=d["description"]
    except Exception:
        miss+=1; continue
    lad[(desc.get("priceLadderDescription") or {}).get("type","?")]+=1
    rate[float(desc.get("marketBaseRate", -1))]+=1
    rt[(desc.get("raceType") or "?")]+=1

print(f"catalogues: {sum(lad.values())} parsed, {miss} unreadable")
print(f"  price ladder : {dict(lad)}")
print(f"  base rate %  : {dict(rate)}")
print(f"  race type    : {dict(rt)}")
non_classic = sum(v for k,v in lad.items() if k!="CLASSIC")
assert non_classic==0, (
    f"{non_classic} markets are not CLASSIC ladder — to_ticks() is wrong for them. "
    "Fix the ladder before using any tick-space result.")
print(f"\n  all CLASSIC: tick math is valid")
print(f"  commission will be read PER MARKET from marketBaseRate "
      f"(median {np.median([k for k,v in rate.items() for _ in range(v)]):.1f}%)")


## Cell 2 — Offline preprocessing
`CACHE_SCHEMA = 6` — identical to V56a.  If V56a already built the cache,
this cell just finds existing `.npz` files and skips rebuilding.

In [ ]:
_SPEC=[(1.01,2.,.01),(2.,3.,.02),(3.,4.,.05),(4.,6.,.1),(6.,10.,.2),(10.,20.,.5),
       (20.,30.,1.),(30.,50.,2.),(50.,100.,5.),(100.,1000.,10.)]
_L=[]
for lo,hi,st in _SPEC: _L.extend(np.round(lo+st*np.arange(int(round((hi-lo)/st))),2))
_L.append(1000.); LADDER=np.array(sorted(set(_L))); _TIX=np.arange(len(LADDER),dtype=float)
NT=len(LADDER)
def to_ticks(p):
    p=np.asarray(p,float); o=np.full(p.shape,np.nan)
    m=np.isfinite(p)&(p>=LADDER[0])&(p<=LADDER[-1])
    if m.any(): o[m]=np.interp(p[m],LADDER,_TIX)
    return o
def tick_px(i): return float(LADDER[int(np.clip(round(i),0,NT-1))])
_t1=lambda x: float(to_ticks(np.array([x]))[0]) if np.isfinite(x) else np.nan

def _lines(path):
    d=zlib.decompressobj(zlib.MAX_WBITS|16); buf=""
    with open(path,"rb") as fh:
        while True:
            c=fh.read(1<<20)
            if not c: break
            try: buf+=d.decompress(c).decode("utf-8","replace")
            except zlib.error: break
            *ls,buf=buf.split("\n")
            for l in ls:
                if l.strip(): yield l

# ---------------------------------------------------------------------------
# V56a-b: distance and class, sized to what the 439-race scan actually showed.
#
# DISTANCE was a linear scalar (dist/2000). But AU distances cluster on a discrete
# grid (~20-30 values) and the microstructure response is not monotone in metres, so
# piecewise-constant buckets are far easier to learn from 434 episodes. Edges are
# CODE-RELATIVE, so a bucket means roughly the same thing (sprint/mile/middle/staying)
# in both codes -- which sidesteps the harness x distance interaction instead of asking
# the network to learn it. dist<=0 encodes all-zero: "unknown", distinguishable from
# "very short" (the old encoding read a parse failure as an extremely short race).
#
# CLASS: the scan showed 19 suffixes, 82% in just three levels (Pace M 201, Hcap 127,
# Mdn 32) and NO benchmark (BM) races at all -- so the fine-grained quality ladder I
# first proposed would never have fired. The whole premium tier is 19/439 races, which
# cannot support one-hotting Grp1/Grp2/Grp3/Listed separately (2-6 races each). They
# collapse to one flag. CL1-CL6 (27 races) stays an ordinal scalar, not a one-hot.
# Pace M / Trot M / 3yo fall through to all-zeros: is_harness and distance cover them.
# ---------------------------------------------------------------------------
import re as _re

def dist_buckets(dist, is_harness):
    """4 dims, code-relative. All-zero = unknown."""
    if not dist or dist <= 0:
        return [0.,0.,0.,0.]
    edges = (1750,2300,2800) if is_harness else (1300,1900,2400)
    b = int(np.digitize(dist, edges))          # 0..3
    return [1.0 if k==b else 0.0 for k in range(4)]

def parse_class(nm):
    """4 dims: [is_mdn, is_hcap, is_premium, cl_level]."""
    m = _re.search(r'\d+m\s+(.*)$', nm or "")
    s = (m.group(1).strip() if m else "").upper()
    is_mdn  = 1.0 if ("MDN" in s or "MAIDEN" in s) else 0.0
    is_hcap = 1.0 if ("HCAP" in s or "HANDICAP" in s) else 0.0
    # Grp1/2/3 + Listed + Qlty + Cup = 19 races total; too thin to separate.
    is_prem = 1.0 if _re.search(r'GRP|\bG[123]\b|LISTED|QLTY|CUP', s) else 0.0
    cl = _re.search(r'\bCL\s*(\d)', s)
    cl_lvl = int(cl.group(1))/6.0 if cl else 0.0
    return [is_mdn, is_hcap, is_prem, cl_lvl]

# is_jumps dropped: the catalogue audit found Harness 225 / Flat 216 and ZERO jumps
# across all 441 races. A constant dim has no variance, and VecNormalize divides by
# the running std -- so it amplifies pure noise and clips it to the bound. Exactly the
# kind of input that drives pre-activations large and saturates a tanh policy.
N_CAT = 14      # 6 market + 4 distance buckets + 4 class

def parse_to_bars(path):
    """-> dict of arrays shaped (T, R, ...) covering the pre-off window only."""
    books=defaultdict(lambda:{"atb":{},"atl":{},"trd":{}})
    ev=defaultdict(list); start_ms=None; inplay_pt=None; gaps=[]
    for line in _lines(path):
        try: m=json.loads(line)
        except json.JSONDecodeError: continue
        op=m.get("op")
        if op=="meta": start_ms=pd.Timestamp(m["market_start_utc"]).timestamp()*1000; continue
        if op=="trailer": inplay_pt=m.get("inplay_from_pt") or inplay_pt; continue
        if op=="gap": gaps.append(m.get("at")); continue
        if op!="mcm": continue
        mc=m["mc"][0]; pt=m.get("pt")
        if mc.get("img"): books.clear()
        if mc.get("marketDefinition"):
            mdf=mc["marketDefinition"]
            if mdf.get("inPlay") and inplay_pt is None: inplay_pt=pt
        if pt is None: continue
        for rc in mc.get("rc",[]):
            sid=rc["id"]; b=books[sid]
            if rc.get("img"): b["atb"],b["atl"],b["trd"]={},{},{}
            bb_pre=max(b["atb"]) if b["atb"] else np.nan
            bl_pre=min(b["atl"]) if b["atl"] else np.nan
            trades=[]
            for px,sz in rc.get("trd",[]):
                prev=b["trd"].get(px,0.0); dv=sz-prev; b["trd"][px]=sz
                if dv>0: trades.append((px,dv))
            for fld in ("atb","atl"):
                for px,sz in rc.get(fld,[]):
                    if sz==0: b[fld].pop(px,None)
                    else:     b[fld][px]=sz
            if not b["atb"] or not b["atl"]: continue
            ev[sid].append((pt, dict(b["atb"]), dict(b["atl"]), dict(b["trd"]),
                            trades, bb_pre, bl_pre))
    if not ev or start_ms is None: return None

    cutoff = inplay_pt if inplay_pt else start_ms
    sids=sorted(ev)
    t0=min(e[0][0] for e in ev.values()); t0=max(t0, cutoff-PREOFF_START_S*1000)
    T=int((cutoff-t0)//(BAR_S*1000))
    if T<60: return None
    # V56a: was sids[:MAX_RUNNERS] -- the FIRST n by selectionId, i.e. arbitrary.
    # Keep the most-traded runners instead; those are the ones with tradeable books.
    n_field=len(sids)
    if len(sids)>MAX_RUNNERS:
        vol={sid: max((e[3] and sum(e[3].values()) or 0.0) for e in ev[sid]) for sid in sids}
        sids=sorted(sorted(sids, key=lambda s_: -vol.get(s_,0.0))[:MAX_RUNNERS])
    R=len(sids)

    F=dict(bbt=np.full((T,R),np.nan), blt=np.full((T,R),np.nan),
           l1b=np.zeros((T,R)), l1l=np.zeros((T,R)),
           t3b=np.zeros((T,R)), t3l=np.zeros((T,R)),
           nearb=np.zeros((T,R)), nearl=np.zeros((T,R)),
           deepb=np.zeros((T,R)), deepl=np.zeros((T,R)),
           tv=np.zeros((T,R)), vwap_t=np.full((T,R),np.nan),
           poc_t=np.full((T,R),np.nan),
           back_agg=np.zeros((T,R)), lay_agg=np.zeros((T,R)),
           flow_lay_ge=np.zeros((T,R,K_OFFSETS+1)),
           flow_back_le=np.zeros((T,R,K_OFFSETS+1)),
           queue_b=np.zeros((T,R,K_OFFSETS+1)), queue_l=np.zeros((T,R,K_OFFSETS+1)))

    for j,sid in enumerate(sids):
        cur=None
        for (pt,atb,atl,trd,trades,bbp,blp) in ev[sid]:
            if pt<t0 or pt>=cutoff: continue
            i=int((pt-t0)//(BAR_S*1000))
            if i>=T: continue
            bb=max(atb); bl=min(atl); bbt=_t1(bb); blt=_t1(bl)
            if not (np.isfinite(bbt) and np.isfinite(blt)): continue
            for px,dv in trades:
                if np.isfinite(bbp) and px<=bbp+1e-9: F["back_agg"][i,j]+=dv
                elif np.isfinite(blp) and px>=blp-1e-9: F["lay_agg"][i,j]+=dv
                pt_=_t1(px)
                if np.isfinite(pt_) and np.isfinite(bbp) and np.isfinite(blp):
                    bbt_p=_t1(bbp); blt_p=_t1(blp)
                    for k in range(K_OFFSETS+1):
                        if pt_>=bbt_p+k-1e-6: F["flow_lay_ge"][i,j,k]+=dv
                        if pt_<=blt_p-k+1e-6: F["flow_back_le"][i,j,k]+=dv
            pxb=np.array(list(atb.keys())); szb=np.array(list(atb.values()))
            pxl=np.array(list(atl.keys())); szl=np.array(list(atl.values()))
            tb,tl=to_ticks(pxb),to_ticks(pxl)
            F["bbt"][i,j]=bbt; F["blt"][i,j]=blt
            F["l1b"][i,j]=atb[bb]; F["l1l"][i,j]=atl[bl]
            F["t3b"][i,j]=szb[np.argsort(-tb)][:3].sum()
            F["t3l"][i,j]=szl[np.argsort(tl)][:3].sum()
            F["nearb"][i,j]=szb[(bbt-tb)<=5].sum(); F["nearl"][i,j]=szl[(tl-blt)<=5].sum()
            F["deepb"][i,j]=szb.sum(); F["deepl"][i,j]=szl.sum()
            if trd:
                tp=np.array(list(trd.keys())); tvv=np.array(list(trd.values()))
                tot=tvv.sum(); F["tv"][i,j]=tot
                if tot>0:
                    F["vwap_t"][i,j]=float(np.average(to_ticks(tp),weights=tvv))
                    F["poc_t"][i,j]=float(to_ticks(np.array([tp[np.argmax(tvv)]]))[0])
            for k in range(K_OFFSETS+1):
                F["queue_b"][i,j,k]=atl.get(tick_px(bbt+k),0.0)   # ahead of our back
                F["queue_l"][i,j,k]=atb.get(tick_px(blt-k),0.0)   # ahead of our lay
            cur=(bbt,blt)
        # forward-fill state columns
    for key in ["bbt","blt","l1b","l1l","t3b","t3l","nearb","nearl","deepb","deepl",
                "tv","vwap_t","poc_t"]:
        A=F[key]
        for j in range(R):
            col=A[:,j]; last=np.nan
            for i in range(T):
                if np.isfinite(col[i]) and col[i]!=0 or (key in ("bbt","blt") and np.isfinite(col[i])):
                    last=col[i]
                elif np.isfinite(last): col[i]=last
    # FIX A: .all(axis=0) required a two-sided book in EVERY bar, so one missing
    # early bar deleted a runner for the whole race -- 10-runner fields became 2.
    # Require 80% coverage and back-fill the leading gap instead.
    for key in ["bbt","blt"]:
        A=F[key]
        for j in range(A.shape[1]):
            c=A[:,j]; fin=np.flatnonzero(np.isfinite(c))
            if len(fin): c[:fin[0]]=c[fin[0]]
    cov=np.isfinite(F["bbt"]).mean(axis=0)
    ok=cov>0.8
    if ok.sum()<2: return None
    for k,v in F.items(): F[k]=v[:,ok] if v.ndim==2 else v[:,ok]
    F["sids"]=np.array(sids)[ok]
    F["race"]=Path(path).name; F["T"]=F["bbt"].shape[0]; F["R"]=F["bbt"].shape[1]
    F["off_pt"]=float(cutoff); F["t0"]=float(t0); F["n_gaps"]=len(gaps)

    # ---- V56a: merge the CATALOGUE -------------------------------------------
    # Market-level regime + a liquidity prior at T-15min, and the per-market
    # commission. Deliberately NOT form/jockey/trainer/pedigree: the market has
    # already priced those and they are in the book the agent observes, and they are
    # high-cardinality categoricals on 434 races -- overfitting fuel.
    cat_path = CAT_DIR/(path.name.replace(".ndjson.gz",".json"))
    cm=np.zeros(N_CAT,dtype=np.float32); comm=COMMISSION
    if cat_path.exists():
        try:
            cat=json.loads(cat_path.read_text())["catalogue"]
            desc,evt=cat["description"],cat["event"]
            rtype=(desc.get("raceType") or "").lower()
            nm=cat.get("marketName","")
            dist=next((int(w[:-1]) for w in nm.split()
                       if w.endswith("m") and w[:-1].isdigit()),0)
            is_h_=any(k in rtype for k in ("harness","pace","trot"))
            dbk=dist_buckets(dist,is_h_)
            ctier=parse_class(nm)
            t_open=pd.Timestamp(evt["openDate"]).timestamp()
            t_off =pd.Timestamp(cat["marketStartTime"]).timestamp()
            cm=np.array([
                1.0 if is_h_ else 0.0,
                np.log1p(float(cat.get("totalMatched") or 0.0))/12.0,  # liquidity prior
                float(bool(desc.get("bspMarket"))),
                float(bool(desc.get("turnInPlayEnabled"))),
                float(bool(desc.get("persistenceEnabled"))),
                np.clip((t_off-t_open)/86400.0,0,3)/3.0,               # market age
            ]+dbk+ctier,dtype=np.float32)
            comm=float(desc.get("marketBaseRate",8.0))/100.0
        except Exception as _e:
            print(f"    catalogue parse failed for {path.name[-24:]}: {_e}")
    else:
        print(f"    NO CATALOGUE for {path.name[-24:]} -- defaults used")
    F["cat_market"]=cm; F["commission"]=float(comm)
    F["n_field"]=int(n_field)          # true field size before the MAX_RUNNERS cut
    return F

In [ ]:
# ---------------------------------------------------------------------------
# Cache keys are VERSIONED. A .npz written under a different MAX_RUNNERS (or a changed
# feature schema) can no longer be silently reused -- it simply isn't found, so it
# rebuilds. Filename-only keys produced "could not broadcast (16,34) into (12,34)"
# two cells downstream three separate times.
# ---------------------------------------------------------------------------
CACHE_SCHEMA = 6        # bump whenever parse_to_bars changes what it emits

def cache_key(p):
    return CACHE_DIR / f"{p.stem}__r{MAX_RUNNERS}_k{K_OFFSETS}_s{CACHE_SCHEMA}.npz"

def build_cache(force=False, quiet_after=10):
    out, built, skipped = [], 0, 0
    files = sorted(REC_DIR.glob("*.ndjson.gz"))
    if FASTTRACK_N:
        _r=np.random.default_rng(FASTTRACK_SEED)
        _k=min(FASTTRACK_N*3, len(files))      # oversample: some races get skipped
        files=[files[i] for i in sorted(_r.choice(len(files),_k,replace=False))]
        print(f"  FAST TRACK: {len(files)} of "
              f"{len(list(REC_DIR.glob('*.ndjson.gz')))} recordings sampled")
    # A/B INTEGRITY: Arm A ran on 460 races, Arm B on 440 -- 20 arrived between runs, so
    # the splits diverged (322/69/69 vs 308/66/66). Stage A survived that (both picked
    # the same earliest race) but Stage B/C would not. Pin the corpus for the duration.
    if CORPUS_FREEZE:
        n0=len(files)
        files=[f for f in files if f.name[:8] <= CORPUS_FREEZE]
        if len(files)<n0:
            print(f"  corpus frozen at {CORPUS_FREEZE}: {len(files)}/{n0} recordings used")
    for n, p in enumerate(files):
        cp = cache_key(p)
        if cp.exists() and not force:
            out.append(cp); continue
        try:
            F = parse_to_bars(p)
            if F is None:
                skipped += 1
                if skipped <= quiet_after: print(f"  skip {p.name[-26:]}")
                continue
            np.savez_compressed(cp, **{k: v for k, v in F.items()
                                       if isinstance(v, (np.ndarray, int, float, str))})
            out.append(cp); built += 1
            if built <= quiet_after:
                print(f"  {p.name[-26:]}  T={F['T']:>4} R={F['R']:>2} "
                      f"field={F.get('n_field','?')} comm={F.get('commission',0):.0%}")
            elif built % 50 == 0:
                print(f"  ... {built} built ({n+1}/{len(files)})", flush=True)
        except Exception as e:
            print(f"  FAIL {p.name[-26:]}: {type(e).__name__} {e}")
    if skipped > quiet_after: print(f"  ... and {skipped-quiet_after} more skipped")

    live = {c.name for c in out}
    stale = [c for c in CACHE_DIR.glob("*.npz") if c.name not in live]
    if stale:
        print(f"  {len(stale)} stale cache files from an earlier config "
              f"(delete with: [c.unlink() for c in stale])")
    return out

# FRESH BUILD over the full ~619-recording corpus. CACHE_SCHEMA is bumped and force=True,
# so every race is re-parsed under the current MAX_RUNNERS / K_OFFSETS / feature schema.
# Nothing from an earlier config can be silently reused. Expect several minutes.
CACHE = build_cache(force=True)
print(f"\n{len(CACHE)} races cached  (key: r{MAX_RUNNERS}_k{K_OFFSETS}_s{CACHE_SCHEMA})")
assert len(CACHE) >= 2, (
    f"Only {len(CACHE)} cached from {n_local} recordings — every file was skipped "
    "by parse_to_bars.")

## Cell 2b — Adverse selection calibration

In [ ]:
def calibrate_adverse(races, horizon_bars=5, min_obs=500):
    """Mean adverse touch-mid move (in ticks) following aggressive flow."""
    obs=[]
    for F in races:
        mid=0.5*(F["bbt"]+F["blt"]); T=F["T"]
        la,ba=F["lay_agg"],F["back_agg"]
        for t in range(T-horizon_bars):
            d=mid[t+horizon_bars]-mid[t]
            m=la[t]>0
            if m.any(): obs.extend(( +1.0*d[m]).tolist())   # lay flow -> odds drift out
            m=ba[t]>0
            if m.any(): obs.extend(( -1.0*d[m]).tolist())   # back flow -> odds shorten
    a=np.array([x for x in obs if np.isfinite(x)])
    if len(a)<min_obs: return None,len(a),np.nan
    return float(np.mean(a)), len(a), float(np.std(a)/np.sqrt(len(a)))

_tmp=[]
for cp in CACHE:
    z=np.load(cp,allow_pickle=True); _tmp.append({k:z[k] for k in z.files})
for F in _tmp: F["T"],F["R"]=int(F["T"]),int(F["R"])

_adv,_n,_se=calibrate_adverse(_tmp)
if _adv is None or not np.isfinite(_adv):
    ADVERSE_TICKS=ADVERSE_TICKS_FALLBACK
    print(f"insufficient flow observations -> falling back to {ADVERSE_TICKS}")
else:
    ADVERSE_TICKS=float(np.clip(abs(_adv),0.25,4.0))
    print(f"measured adverse selection: {_adv:+.3f} ticks (+/-{1.96*_se:.3f}, n={_n:,})")
    print(f"ADVERSE_TICKS = {ADVERSE_TICKS:.2f}   (V55 guessed 1.00)")
    if abs(_adv)<0.1:
        print("  WARNING: near zero. Either flow carries no information, or the")
        print("  sign convention in parse_to_bars is inverted. Check before training.")
del _tmp

## Cell 3 — Observation spec (V57)

26 static runner features (identical to V56a) + **6 own-position features** (new).
Market features: V56a's 24 + 2 portfolio-level dims = 26.

`OBS_DIM = MAX_RUNNERS × NF_RUN + len(MKT_FEATS) + MAX_RUNNERS = 12×32 + 26 + 12 = 422`

In [ ]:
# ── Static runner features (identical to V56a) ───────────────────────────────
RUNNER_FEATS = [
 "spread_t","wom_l1","wom_l3","wom_near","wom_deep","shape_b","shape_l",
 "log_depth","log_tv","ofi_1s","ofi_5s","ofi_15s","trade_int",
 "vwap_dist","poc_dist","ret_5s","ret_15s","ret_60s","field_rel_ret",
 "field_rel_ofi","spread_rel",
 "prob","prob_rank","prob_gap",
 "own_back_off","own_lay_off"]   # last 2 zeros in V57 (no open passive orders)

# ── Own-position features (V57 — tracks market-order positions) ──────────────
OWN_FEATS = [
 "pos_back",   # back stake / MAX_RUNNER_EXP
 "pos_lay",    # lay liability / MAX_RUNNER_EXP
 "bpx_dist",   # (avg_back_px - current_bb) / current_bb  (+ = backed high)
 "lpx_dist",   # (avg_lay_px  - current_bb) / current_bb  (+ = laid above market)
 "green_r",    # per-runner green value / MAX_RUNNER_EXP
 "pnl_r",      # same for now (placeholder for net realized PnL)
]

# ── Market features (V56a's 24 + 2 portfolio dims) ───────────────────────────
MKT_FEATS_BASE = [
 "secs_to_off_n","log_secs_to_off","log_matched","back_or","lay_or",
 "entropy","n_run_n",
 "is_harness","liq_prior",
 "bsp_mkt","turn_inplay","persist","mkt_age",
 "d_sprint","d_mile","d_middle","d_stay",
 "cls_mdn","cls_hcap","cls_prem","cls_level"]
# V56a has 24; use the same list but store separately so the extractor knows the split
MKT_FEATS = MKT_FEATS_BASE + ["total_risk_frac", "available_frac"]

NF_STATIC = len(RUNNER_FEATS)    # 26
NF_OWN    = len(OWN_FEATS)       # 6
NF_RUN    = NF_STATIC + NF_OWN   # 32
OBS_DIM   = MAX_RUNNERS * NF_RUN + len(MKT_FEATS) + MAX_RUNNERS  # 422

print(f"NF_STATIC  = {NF_STATIC}")
print(f"NF_OWN     = {NF_OWN}")
print(f"NF_RUN     = {NF_RUN}")
print(f"MKT_FEATS  = {len(MKT_FEATS)}")
print(f"OBS_DIM    = {OBS_DIM}")

# ── precompute_features: IDENTICAL to V56a ───────────────────────────────────
# (own_back_off / own_lay_off are set to 0 since there are no passive orders;
#  the OWN_FEATS above fill those slots dynamically in _obs())
def precompute_features(F):
    """Static (market-derived) part of the observation. Own-order dims stay zero."""
    T,R=F["T"],F["R"]; eps=1e-9
    bbt,blt=F["bbt"],F["blt"]
    mid=0.5*(bbt+blt)
    X=np.zeros((T,R,NF_STATIC),dtype=np.float32)
    g=lambda n: RUNNER_FEATS.index(n)
    X[:,:,g("spread_t")]=np.clip(blt-bbt,0,40)/10.0
    for nm,(b,l) in [("wom_l1",(F["l1b"],F["l1l"])),("wom_l3",(F["t3b"],F["t3l"])),
                     ("wom_near",(F["nearb"],F["nearl"])),("wom_deep",(F["deepb"],F["deepl"]))]:
        X[:,:,g(nm)]=(b-l)/(b+l+eps)
    X[:,:,g("shape_b")]=F["t3b"]/(F["deepb"]+eps)
    X[:,:,g("shape_l")]=F["t3l"]/(F["deepl"]+eps)
    X[:,:,g("log_depth")]=np.log1p(F["deepb"]+F["deepl"])/10.0
    X[:,:,g("log_tv")]=np.log1p(F["tv"])/10.0
    ofi=F["lay_agg"]-F["back_agg"]; tot=F["lay_agg"]+F["back_agg"]
    X[:,:,g("ofi_1s")]=np.tanh(ofi/50.0)
    for w,nm in [(5,"ofi_5s"),(15,"ofi_15s")]:
        c=np.cumsum(ofi,axis=0)
        roll=c-np.vstack([np.zeros((min(w,T),R)),c[:-w]]) if T>w else c
        X[:,:,g(nm)]=np.tanh(roll/(50.0*w))
    X[:,:,g("trade_int")]=np.log1p(tot)/5.0
    X[:,:,g("vwap_dist")]=np.nan_to_num(np.clip(F["vwap_t"]-mid,-20,20))/10.0
    X[:,:,g("poc_dist")]=np.nan_to_num(np.clip(F["poc_t"]-mid,-20,20))/10.0
    for w,nm in [(5,"ret_5s"),(15,"ret_15s"),(60,"ret_60s")]:
        r=np.zeros((T,R)); r[w:]=mid[w:]-mid[:-w]
        X[:,:,g(nm)]=np.clip(r,-15,15)/5.0
    fr=X[:,:,g("ret_15s")]
    X[:,:,g("field_rel_ret")]=fr-fr.mean(axis=1,keepdims=True)
    o1=X[:,:,g("ofi_1s")]
    X[:,:,g("field_rel_ofi")]=o1-o1.mean(axis=1,keepdims=True)
    sp=np.clip(blt-bbt,0,40)
    c=np.cumsum(sp,axis=0); w=60
    roll=(c-np.vstack([np.zeros((min(w,T),R)),c[:-w]]))/min(w,T) if T>w else sp
    X[:,:,g("spread_rel")]=np.tanh(sp/np.maximum(roll,1e-9)-1.0)
    px=0.5*(LADDER[np.clip(bbt.round().astype(int),0,NT-1)]
            +LADDER[np.clip(blt.round().astype(int),0,NT-1)])
    prob=1.0/np.maximum(px,1.01); pn=prob/np.maximum(prob.sum(axis=1,keepdims=True),eps)
    X[:,:,g("prob")]=pn
    X[:,:,g("prob_rank")]=np.argsort(np.argsort(-pn,axis=1),axis=1)/max(R-1,1)
    X[:,:,g("prob_gap")]=pn.max(axis=1,keepdims=True)-pn
    # own_back_off, own_lay_off stay 0 (no passive orders in V57)
    M=np.zeros((T,len(MKT_FEATS)),dtype=np.float32)
    k=lambda n: MKT_FEATS.index(n)
    t_vec=np.arange(T-1,-1,-1,dtype=float)*BAR_S
    M[:,k("secs_to_off_n")]=np.clip(t_vec/PREOFF_START_S,0,1)
    M[:,k("log_secs_to_off")]=np.log1p(t_vec)/np.log1p(PREOFF_START_S)
    if "tv_mkt" in F: M[:,k("log_matched")]=np.log1p(F["tv_mkt"])/15.0
    if "back_or" in F: M[:,k("back_or")]=np.clip((F["back_or"]-1.0)/0.2,0,1)
    if "lay_or"  in F: M[:,k("lay_or")] =np.clip((F["lay_or"]-1.0)/0.2,0,1)
    M[:,k("entropy")]=(-pn*np.log(pn+eps)).sum(axis=1)/math.log(max(R,2))
    M[:,k("n_run_n")]=R/MAX_RUNNERS
    for kn,fk in [("is_harness","is_harness"),("liq_prior","liq_prior"),
                  ("bsp_mkt","bsp_mkt"),("turn_inplay","turn_inplay"),
                  ("persist","persist"),("mkt_age","mkt_age")]:
        if fk in F: M[:,k(kn)]=float(F[fk])
    for kn in ("d_sprint","d_mile","d_middle","d_stay"):
        if kn in F: M[:,k(kn)]=float(F[kn])
    for kn in ("cls_mdn","cls_hcap","cls_prem","cls_level"):
        if kn in F: M[:,k(kn)]=float(F[kn])
    # portfolio dims filled dynamically in _obs(); leave as zero here
    return X, M

print("precompute_features defined (V57 — static block)")


## Cell 3b — Load & precompute races

In [ ]:
import random as _rnd

def load_races(cache_paths, n=None, seed=SEED):
    rng = np.random.default_rng(seed)
    paths = list(cache_paths)
    if n is not None:
        paths = list(rng.choice(paths, min(n, len(paths)), replace=False))
    races = []
    for cp in paths:
        z = np.load(cp, allow_pickle=True)
        F = {k: z[k] for k in z.files}
        F["T"] = int(F["T"]); F["R"] = int(F["R"])
        # Build mask (T, MAX_RUNNERS)
        R = F["R"]
        mask = np.zeros((F["T"], MAX_RUNNERS), dtype=np.float32)
        mask[:, :R] = 1.0
        F["mask"] = mask
        # Precompute static features
        F["X"], F["M_static"] = precompute_features(F)
        races.append(F)
    return races

# Fast-track: load N random races only
print(f"Loading {FASTTRACK_N} random races for Stage A...")
RACES_ALL = load_races(CACHE, n=FASTTRACK_N, seed=SEED)
print(f"Loaded {len(RACES_ALL)} races")
for F in RACES_ALL:
    print(f"  {F.get('name','?')}  T={F['T']} R={F['R']} comm={F.get('commission',COMMISSION):.0%}")


## Cell 4 — Environment (V57 — 25-action directional spec)

**Action[0:MAX_RUNNERS]**: per-runner signal ∈ [-1,1].  Positive = back, negative = lay.
**Action[MAX_RUNNERS]**: stake fraction → (x+1)/2 ∈ [0,1] of available capital.

Allocation per runner = `total_wager × |signal[r]| / Σ|signal|`.
Fill: back at `bl` (cross the spread to take lay offers), lay at `bb`.

Reward: potential-based shaping Φ(s) = `net_green(book)`.
`r_t = γΦ(s') − Φ(s)`.  Terminal: forced flatten at off prices.

In [ ]:
import gymnasium as gym
from gymnasium import spaces

# ── commission helper (identical to V56a) ────────────────────────────────────
def green_value(pos_back_stake, pos_back_px, pos_lay_liab, pos_lay_px, bb, bl):
    out = np.zeros_like(bb)
    m = pos_back_stake > 0
    if m.any(): out[m] += pos_back_stake[m] * (pos_back_px[m] / np.maximum(bl[m], 1.01) - 1.0)
    m = pos_lay_liab > 0
    if m.any(): out[m] += pos_lay_liab[m] * (1.0 - pos_lay_px[m] / np.maximum(bb[m], 1.01))
    return out

def net_green(gross_per_runner, comm):
    g = float(np.sum(gross_per_runner))
    return g * (1.0 - comm) if g > 0 else g


class DirectionalRaceEnv(gym.Env):
    """V57: 25-action directional trading environment.

    The single stake-fraction dimension solves V56a 24-dim coordination problem:
    to do nothing, the agent only needs action[MAX_RUNNERS] -> -1 (stake->0).
    """
    metadata = {"render_modes": []}

    def __init__(self, races, difficulty=1.0, null_mode=False, seed=SEED):
        super().__init__()
        self.races      = races
        self.difficulty = difficulty
        self.null_mode  = null_mode
        self.rng        = np.random.default_rng(seed)
        self.action_space      = spaces.Box(-1, 1, shape=(MAX_RUNNERS + 1,), dtype=np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(OBS_DIM,), dtype=np.float32)

    # ── reset ────────────────────────────────────────────────────────────────
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.F    = self.races[self.rng.integers(len(self.races))]
        self.T    = self.F["T"]; self.R = self.F["R"]
        self.comm = float(self.F.get("commission", COMMISSION))
        self.i    = 0
        self.bs   = np.zeros(MAX_RUNNERS)   # back stake
        self.bpx  = np.zeros(MAX_RUNNERS)   # avg back fill price
        self.ll   = np.zeros(MAX_RUNNERS)   # lay liability
        self.lpx  = np.zeros(MAX_RUNNERS)   # avg lay fill price
        self.prev_phi     = 0.0
        self.shaping_total= 0.0
        self.n_trades     = 0
        self.bk_px_sum    = np.zeros(MAX_RUNNERS)
        self.bk_px_n      = np.zeros(MAX_RUNNERS)
        self.ly_px_sum    = np.zeros(MAX_RUNNERS)
        self.ly_px_n      = np.zeros(MAX_RUNNERS)
        return self._obs(), {}

    # ── price helpers ─────────────────────────────────────────────────────────
    def _px(self, arr, i=None):
        i = self.i if i is None else i
        return LADDER[np.clip(np.round(arr[i]).astype(int), 0, NT-1)]

    def _px_pad(self, arr, i=None):
        """Padded price array → shape (MAX_RUNNERS,); out-of-range runners = 1.01."""
        raw = self._px(arr, i)            # shape (R,)
        out = np.full(MAX_RUNNERS, 1.01)
        out[:len(raw)] = raw
        return out

    # ── observation ──────────────────────────────────────────────────────────
    def _obs(self):
        i    = min(self.i, self.T - 1)
        X    = self.F["X"][i]             # (R, NF_STATIC)
        M_st = self.F["M_static"][i]      # (len(MKT_FEATS),) precomputed
        mask = self.F["mask"][i]          # (MAX_RUNNERS,)
        bb   = self._px_pad(self.F["bbt"], i)
        bl   = self._px_pad(self.F["blt"], i)

        # Build runner block (MAX_RUNNERS, NF_RUN)
        obs_r = np.zeros((MAX_RUNNERS, NF_RUN), dtype=np.float32)
        obs_r[:self.R, :NF_STATIC] = X   # static features

        # Own-position features (6 per runner)
        for r in range(self.R):
            obs_r[r, NF_STATIC + 0] = self.bs[r] / MAX_RUNNER_EXP
            obs_r[r, NF_STATIC + 1] = self.ll[r] / MAX_RUNNER_EXP
            if self.bs[r] > 0 and bb[r] > 1.0:
                obs_r[r, NF_STATIC + 2] = np.clip((self.bpx[r] - bb[r]) / bb[r], -1, 1)
            if self.ll[r] > 0 and bb[r] > 1.0:
                obs_r[r, NF_STATIC + 3] = np.clip((self.lpx[r] - bb[r]) / bb[r], -1, 1)
            gv = float(green_value(
                np.array([self.bs[r]]), np.array([self.bpx[r]]),
                np.array([self.ll[r]]), np.array([self.lpx[r]]),
                np.array([bb[r]]),      np.array([bl[r]]))[0])
            obs_r[r, NF_STATIC + 4] = np.clip(gv / max(MAX_RUNNER_EXP, 1), -1, 1)
            obs_r[r, NF_STATIC + 5] = obs_r[r, NF_STATIC + 4]

        obs_r *= mask[:, np.newaxis]

        # Portfolio dims (fill the last 2 slots of M_static in-place copy)
        M = M_st.copy()
        total_risk       = (self.bs.sum() + self.ll.sum()) / STARTING_BANK
        M[-2]            = np.float32(min(total_risk, 1.0))
        M[-1]            = np.float32(max(0.0, 1.0 - total_risk))

        return np.concatenate([obs_r.reshape(-1), M, mask]).astype(np.float32)

    # ── step ─────────────────────────────────────────────────────────────────
    def step(self, action):
        signals    = np.clip(action[:MAX_RUNNERS], -1.0, 1.0).astype(float)
        stake_frac = (float(action[MAX_RUNNERS]) + 1.0) / 2.0  # [-1,1] → [0,1]

        i    = min(self.i, self.T - 1)
        mask = self.F["mask"][i].astype(bool)
        bb   = self._px_pad(self.F["bbt"])   # lay-order fill prices for backs
        bl   = self._px_pad(self.F["blt"])   # back-order fill prices for lays

        # Null-control: permute signals so they carry no per-runner information
        if self.null_mode:
            signals = self.rng.permutation(signals)

        # Only trade on valid runners
        masked_sig = signals.copy()
        masked_sig[~mask] = 0.0

        # Available capital and wager
        total_risk   = self.bs.sum() + self.ll.sum()
        available    = max(0.0, STARTING_BANK - total_risk)
        total_wager  = available * stake_frac

        abs_sig  = np.abs(masked_sig)
        abs_sum  = abs_sig.sum()

        if abs_sum > 1e-8 and total_wager >= MIN_BET:
            weights = abs_sig / abs_sum
            alloc   = total_wager * weights

            for r in range(MAX_RUNNERS):
                if alloc[r] < MIN_BET or not mask[r]:
                    continue
                sig = float(masked_sig[r])
                a   = float(alloc[r])

                if sig > 0:          # ── BACK at bl[r] ──────────────────────
                    px = float(bl[r])
                    if px < 1.01: continue
                    if self.bs[r] > 0:
                        self.bpx[r] = (self.bs[r]*self.bpx[r] + a*px) / (self.bs[r]+a)
                    else:
                        self.bpx[r] = px
                    self.bs[r]      += a
                    self.n_trades   += 1
                    self.bk_px_sum[r] += px; self.bk_px_n[r] += 1

                elif sig < 0:        # ── LAY at bb[r]  ──────────────────────
                    px   = float(bb[r])
                    if px < 1.01: continue
                    liab = a
                    if self.ll[r] > 0:
                        self.lpx[r] = (self.ll[r]*self.lpx[r] + liab*px) / (self.ll[r]+liab)
                    else:
                        self.lpx[r] = px
                    self.ll[r]      += liab
                    self.n_trades   += 1
                    self.ly_px_sum[r] += px; self.ly_px_n[r] += 1

        # Potential-based shaping reward
        phi    = net_green(green_value(self.bs, self.bpx, self.ll, self.lpx, bb, bl), self.comm)
        reward = GAMMA * phi - self.prev_phi
        self.prev_phi       = phi
        self.shaping_total += reward

        self.i += 1
        done   = self.i >= self.T
        info   = {}

        if done:
            i_fin = self.T - 1
            bb_f  = self._px_pad(self.F["bbt"], i_fin)
            bl_f  = self._px_pad(self.F["blt"], i_fin)
            final_green = net_green(
                green_value(self.bs, self.bpx, self.ll, self.lpx, bb_f, bl_f), self.comm)
            reward += final_green - self.prev_phi   # terminal correction
            info = {
                "final_green"   : final_green,
                "n_trades"      : self.n_trades,
                "shaping_total" : self.shaping_total,
                "bk_px_sum"     : self.bk_px_sum.copy(),
                "bk_px_n"       : self.bk_px_n.copy(),
                "ly_px_sum"     : self.ly_px_sum.copy(),
                "ly_px_n"       : self.ly_px_n.copy(),
                "n_runners"     : int(self.R),
            }

        return self._obs(), float(reward), done, False, info

print("DirectionalRaceEnv defined")

# ── quick sanity: one random-policy episode ───────────────────────────────────
_e = DirectionalRaceEnv(RACES_ALL); obs, _ = _e.reset(); done = False; steps = 0
while not done:
    a   = _e.action_space.sample()
    obs, r, done, _, info = _e.step(a); steps += 1
print(f"  sanity: T={steps}  final_green=${info.get('final_green',0):.2f}"
      f"  n_trades={info.get('n_trades',0)}")

# ── do-nothing baseline: action[MAX_RUNNERS]=-1 → stake_frac=0 ──────────────
_dn_rewards = []
for _ in range(20):
    obs, _ = _e.reset(); done = False; ep_r = 0
    while not done:
        a = np.zeros(MAX_RUNNERS + 1, dtype=np.float32)
        a[MAX_RUNNERS] = -1.0          # stake_frac = 0
        obs, r, done, _, info = _e.step(a); ep_r += r
    _dn_rewards.append(info.get("final_green", 0))
print(f"  do-nothing check: mean=${np.mean(_dn_rewards):.4f}  (should be ~0.00)")


## Cell 5 — SAC with equivariant encoder (V57)

Re-uses V56a's equivariant DeepSets extractor, updated for V57's observation dimensions
(`NF_RUN=32`, `MKT_FEATS=26`).  The action space is now 25-dim.

`ENT_COEF_FLOOR` is raised to 0.05 to prevent the entropy collapse observed in V56a.

In [ ]:
import torch, torch.nn as nn
from stable_baselines3 import SAC
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.sac.policies import SACPolicy, Actor as _SB3Actor

# ── Phantom-runner action masking ─────────────────────────────────────────────
# For a race with R < MAX_RUNNERS active runners, action dims [R:MAX_RUNNERS]
# are meaningless (no runner to trade on).  Without masking, SAC spends entropy
# budget and gradient steps on these phantom dims.
#
# Fix: inside the actor, override mean→0 and log_std→PHANTOM_LOG_STD for phantom
# dims so they are near-deterministic (std ≈ 0.05) and contribute ~zero entropy.
#
# How the mask is recovered from normalised obs
# After VecNormalize the binary mask (0/1) normalises so active > 0 and phantom < 0
# for any typical runner distribution (mean_mask in (0,1)).  Threshold > 0 is robust.
#
# Entropy target note
# Each clamped phantom dim contributes H ≈ 0.5*log(2πe) + PHANTOM_LOG_STD ≈ -1.58 nats.
# For MAX_RUNNERS=12 with R_avg=10: phantom offset ≈ 2 × -1.58 = -3.2 nats total.
# Monitor "H_eff" printed by Metrics (active-only entropy) to tune TARGET_ENTROPY.

PHANTOM_LOG_STD = -3.0       # std ≈ 0.05; H per phantom dim ≈ -1.58 nats
_MASK_SLICE     = slice(MAX_RUNNERS * NF_RUN + len(MKT_FEATS),
                        MAX_RUNNERS * NF_RUN + len(MKT_FEATS) + MAX_RUNNERS)
_H_phantom      = 0.5 * float(np.log(2 * np.pi * np.e)) + PHANTOM_LOG_STD   # ≈ -1.58


class DeepSetsExtractor(BaseFeaturesExtractor):
    """Equivariant per-runner embedding + broadcast field context (V57).

    Identical to V56a Arm B but updated for NF_RUN=32 and len(MKT_FEATS)=26.
    The actor still emits 25 dims: 24 runner signals + 1 stake fraction.
    """
    def __init__(self, obs_space, per_dim=32, ctx_dim=64, features_dim=None):
        super().__init__(obs_space, MAX_RUNNERS * (per_dim + ctx_dim))
        self.per_dim, self.ctx_dim = per_dim, ctx_dim
        self.per = nn.Sequential(nn.Linear(NF_RUN, 128), nn.ReLU(),
                                 nn.Linear(128, per_dim), nn.ReLU())
        self.ctx = nn.Sequential(
            nn.Linear(per_dim * 2 + len(MKT_FEATS), ctx_dim), nn.ReLU())

    def forward(self, x):
        B = x.shape[0]; n = MAX_RUNNERS * NF_RUN
        R    = x[:, :n].view(B, MAX_RUNNERS, NF_RUN)
        M    = x[:, n : n + len(MKT_FEATS)]
        mask = x[:, n + len(MKT_FEATS):].unsqueeze(-1)
        h    = self.per(R) * mask
        mean = h.sum(1) / mask.sum(1).clamp(min=1)
        mx   = h.masked_fill(mask == 0, -1e30).max(1).values
        mx   = torch.nan_to_num(mx, neginf=0.0)
        c    = self.ctx(torch.cat([mean, mx, M], dim=1))
        out  = torch.cat([h, c.unsqueeze(1).expand(-1, MAX_RUNNERS, -1)], dim=-1)
        return (out * mask).reshape(B, -1)


class MaskedActor(_SB3Actor):
    """SAC Actor that constrains phantom runner dims to a near-deterministic distribution.

    Phantom runners (index >= R_active) receive mean=0 and log_std=PHANTOM_LOG_STD,
    producing actions indistinguishable from zero.  Active runner dims and the
    stake-fraction dim (index MAX_RUNNERS) are left entirely unchanged.

    Why this works
    - Entropy term: phantom dims contribute ~zero entropy, so the SAC entropy
      objective scales naturally with the number of active runners.
    - Policy gradients: near-deterministic phantom dims carry near-zero gradient
      signal back through the actor — no wasted network capacity.
    - No reward penalty needed: the distribution itself is the constraint.
    """

    def get_action_dist_params(self, obs: torch.Tensor):
        mean, log_std, kwargs = super().get_action_dist_params(obs)

        # Recover binary runner mask from normalised obs (active > 0, phantom <= 0)
        runner_mask = obs[:, _MASK_SLICE] > 0.0          # (B, MAX_RUNNERS)  bool

        # Runner signal dims: clamp phantoms
        r_mean    = torch.where(runner_mask,
                                mean[:, :MAX_RUNNERS],
                                torch.zeros_like(mean[:, :MAX_RUNNERS]))
        r_log_std = torch.where(runner_mask,
                                log_std[:, :MAX_RUNNERS],
                                obs.new_full((obs.shape[0], MAX_RUNNERS), PHANTOM_LOG_STD))

        # Stake-fraction dim (MAX_RUNNERS) always active — concat unchanged
        mean    = torch.cat([r_mean,    mean[:, MAX_RUNNERS:MAX_RUNNERS + 1]], dim=1)
        log_std = torch.cat([r_log_std, log_std[:, MAX_RUNNERS:MAX_RUNNERS + 1]], dim=1)
        # Cache pre-tanh Gaussian entropy for the Metrics callback (lightweight, no alloc)
        with torch.no_grad():
            self._last_entropy = (0.5 * float(np.log(2 * np.pi * np.e)) +
                                  log_std.mean(-1)).mean().item()
        return mean, log_std, kwargs


class MaskedSACPolicy(SACPolicy):
    """SACPolicy that swaps in MaskedActor for phantom-runner masking."""

    def make_actor(self, features_extractor=None):
        actor_kwargs = self._update_features_extractor(self.actor_kwargs, features_extractor)
        return MaskedActor(**actor_kwargs).to(self.device)


class Metrics(BaseCallback):
    """Enforces entropy floor + ceiling; logs episode metrics including active-runner entropy."""
    def __init__(self): super().__init__(); self.rows = []

    def _on_step(self):
        if hasattr(self.model, "log_ent_coef") and self.model.log_ent_coef is not None:
            with torch.no_grad():
                self.model.log_ent_coef.clamp_(
                    float(np.log(ENT_COEF_FLOOR)),
                    float(np.log(ENT_COEF_CEIL)))
        for info in self.locals.get("infos", []):
            if "final_green" in info:
                self.rows.append(info)
        if len(self.rows) and len(self.rows) % 25 == 0 and self.rows[-1].get("_p") != 1:
            self.rows[-1]["_p"] = 1
            d  = pd.DataFrame(self.rows[-25:])
            ec = (self.model.log_ent_coef.exp().item()
                  if getattr(self.model, "log_ent_coef", None) is not None else float("nan"))
            green_pct = int(100 * (d["final_green"] > 0).mean())
            pnl_mean  = d["final_green"].mean()
            trades    = d["n_trades"].mean() if "n_trades" in d else float("nan")
            H_raw     = getattr(self.model.actor, "_last_entropy", float("nan"))
            # Estimate active-only entropy by removing average phantom contribution
            r_avg     = d["n_runners"].mean() if "n_runners" in d else MAX_RUNNERS
            n_phantom = max(0.0, MAX_RUNNERS - r_avg)
            H_eff     = H_raw - n_phantom * _H_phantom   # entropy from active dims only
            ep        = len(self.rows)
            print(f"  ep {ep:>5} | green {green_pct:>3}% | pnl ${pnl_mean:>8.2f}"
                  f" | trades {trades:>6.1f} | ent {ec:.4f}"
                  f" | H_raw {H_raw:.1f} | H_eff {H_eff:.1f}"
                  f" | R_avg {r_avg:.1f}")
        return True


def make_model(env, seed=SEED, target_entropy=TARGET_ENTROPY):
    # Device: CUDA is ~50-100x faster than CPU for SAC training.
    # If this prints "cpu" go to Colab Runtime -> Change runtime type -> GPU.
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    if _device == "cpu":
        print("  WARNING: device=cpu — training will be very slow (~2-5 fps).")
        print("  Go to Runtime -> Change runtime type -> GPU then re-run from Cell 0b.")
    else:
        print(f"  device: {_device}  ({torch.cuda.get_device_name(0)})")

    venv = DummyVecEnv([lambda: env])
    venv = VecNormalize(venv, norm_obs=True, norm_reward=True,
                        clip_obs=10.0, clip_reward=10.0, gamma=GAMMA)
    extractor_kwargs = dict(per_dim=32, ctx_dim=64)
    policy_kwargs = dict(
        features_extractor_class  = DeepSetsExtractor,
        features_extractor_kwargs = extractor_kwargs,
        net_arch                  = [256, 256],
        log_std_init              = -2.0,
    )
    model = SAC(
        MaskedSACPolicy,        # injects phantom-runner action masking
        venv,
        learning_rate   = 3e-4,
        buffer_size     = 50_000,
        learning_starts = 500,
        batch_size      = 256,
        tau             = 0.01,
        gamma           = GAMMA,
        train_freq      = 4,    # step env 4x then train — better CPU/GPU overlap
        gradient_steps  = 4,    # same total gradient steps, less per-step overhead
        ent_coef        = "auto",
        target_entropy  = target_entropy,
        policy_kwargs   = policy_kwargs,
        device          = _device,
        verbose         = 0,
        seed            = seed,
    )
    return model, venv

print("SAC + DeepSetsExtractor + MaskedActor defined")
print(f"  action dims      : {MAX_RUNNERS + 1}  ({MAX_RUNNERS} runner signals + 1 stake fraction)")
print(f"  OBS_DIM          : {OBS_DIM}")
print(f"  features_dim     : {MAX_RUNNERS * (32 + 64)}")
print(f"  PHANTOM_LOG_STD  : {PHANTOM_LOG_STD}  (std = {np.exp(PHANTOM_LOG_STD):.4f})")
print(f"  H per phantom dim: {_H_phantom:.2f} nats  (cf. random Gaussian = +1.42)")
print(f"  CUDA available   : {torch.cuda.is_available()}")


## Stage A — Single-race overfit test

Identical gate to V56a FastTrack:
- random  ~  −$200   (untrained flailing)
- do-nothing = $0    (floor any competent policy must reach)
- trained ≥ −$1      → pass (found no-trade optimum)
- trained > $0       → real edge detected

With the V57 action spec, the do-nothing policy requires only one action dimension
(`action[24] → −1`, stake→0) rather than V56a's 24-dimensional coordination.


In [ ]:
def rollout(env, policy=None, n=20, seed=0, deterministic=True):
    rng  = np.random.default_rng(seed)
    pnls = []
    for _ in range(n):
        obs, _ = env.reset(seed=int(rng.integers(1 << 31)))
        done = False; ep_r = 0
        while not done:
            if policy is None:
                a = env.action_space.sample()
            else:
                obs_t = np.array([obs])
                a, _  = policy.predict(obs_t, deterministic=deterministic)
                a     = a[0]
            obs, r, done, _, info = env.step(a)
            ep_r += r
        pnls.append(info.get("final_green", ep_r))
    return np.array(pnls)


def direction_audit(env, policy, n=10):
    bs = ls = bn = ln = 0.0; both = []
    for _ in range(n):
        obs, _ = env.reset(); done = False
        while not done:
            a, _ = policy.predict(np.array([obs]), deterministic=True)
            obs, r, done, _, info = env.step(a[0])
        bps = info["bk_px_sum"]; bpn = info["bk_px_n"]
        lps = info["ly_px_sum"]; lpn = info["ly_px_n"]
        bs += bps.sum(); bn += bpn.sum()
        ls += lps.sum(); ln += lpn.sum()
        m = (bpn > 0) & (lpn > 0)
        if m.any():
            both.extend(((bps[m]/bpn[m]) / (lps[m]/lpn[m])).tolist())
    bk_avg = bs / max(bn, 1); lk_avg = ls / max(ln, 1)
    dir_ok = "CORRECT (back high / lay low)" if bk_avg > lk_avg else "WRONG DIRECTION"
    print(f"  direction: back fills @ {bk_avg:.3f} vs lay fills @ {lk_avg:.3f} -> {dir_ok}")
    if both:
        edge = np.mean([x - 1 for x in both])
        pos  = int(100 * np.mean([x > 1 for x in both]))
        print(f"  two-sided edge on {len(both)} runner-eps: {edge:.4f} ({pos}% positive)")


# ── Stage A overrides ────────────────────────────────────────────────────────
# Commission 0%: remove cold-start barrier so agent can find directional
# signals before confronting the 8% commission hurdle.
# TARGET_ENTROPY raised to 12.0 (was -8.0) so alpha equilibrates in the
# middle of [ENT_COEF_FLOOR, ENT_COEF_CEIL] rather than pinning to the
# floor.  With log_std_init=-2 and 13 action dims, H_initial ~ -7.5 nats;
# target=12.0 > -7.5 so alpha rises from the start, holding the policy at
# std ~ 0.7 per active dim (moderate exploration).
COMM_STAGE_A           = 0.0    # override: no commission for Stage A
TARGET_ENTROPY_STAGE_A = 12.0   # raised; alpha will equilibrate ~0.2-0.5
print(f"  Stage A: COMMISSION={COMM_STAGE_A:.0%}  TARGET_ENTROPY={TARGET_ENTROPY_STAGE_A:+.1f}")
print("  (real commission re-applied from Stage B onwards)")

# ── fps probe ─────────────────────────────────────────────────────────────────
_probe_env = DirectionalRaceEnv(RACES_ALL[:1])
_pm, _pv   = make_model(_probe_env)
_t0        = time.time()
_pm.learn(total_timesteps=2000)
_fps       = 2000 / (time.time() - _t0)
_T         = int(np.mean([r["T"] for r in RACES_ALL]))
_tot       = PASSES_PER_RACE * _T * len(RACES_ALL)
print(f"{len(RACES_ALL)} races x {PASSES_PER_RACE} passes x T~{_T} = {_tot:,} steps")
print(f"  MEASURED {_fps:.1f} fps -> full run ~{_tot/_fps/3600:.1f} h")
if _fps < 20:
    print("  *** SLOW — check device=cuda ***")
del _pm, _pv, _probe_env

# ── mechanical reward-direction check ─────────────────────────────────────────
_chk = DirectionalRaceEnv(RACES_ALL)
def _g(bs, bpx, ll, lpx, bb, bl, comm=COMMISSION):
    return net_green(green_value(np.array([bs]), np.array([bpx]),
                                 np.array([ll]), np.array([lpx]),
                                 np.array([bb]), np.array([bl])), comm)
assert _g(5, 6.0, 0, 0, 5.0, 5.2)  > 0, "back then shorten should be +ve"
assert _g(5, 5.0, 0, 0, 6.0, 6.2)  < 0, "back then drift should be -ve"
assert _g(0, 0, 5, 5.0, 6.0, 6.2)  > 0, "lay then drift should be +ve"
assert _g(0, 0, 5, 6.0, 5.0, 5.2)  < 0, "lay then shorten should be -ve"
print("  reward direction confirmed: BACK HIGH / LAY LOW is what pays")

# ── do-nothing baseline check ─────────────────────────────────────────────────
_dn = []
for _ in range(30):
    obs, _ = _chk.reset(); done = False
    while not done:
        a = np.zeros(MAX_RUNNERS + 1, dtype=np.float32); a[-1] = -1.0
        obs, r, done, _, info = _chk.step(a)
    _dn.append(info.get("final_green", 0))
print(f"  do-nothing baseline: mean=${np.mean(_dn):.4f}  std=${np.std(_dn):.4f}")

# ── Stage A: fresh model per race ─────────────────────────────────────────────
STAGE_A_RESULTS = []

for race_idx, F in enumerate(RACES_ALL):
    name = str(F.get("name", f"race_{race_idx}")).split("/")[-1]
    print()
    print("#" * 72)
    print(f"# RACE {race_idx+1}/{len(RACES_ALL)}  {name}"
          f"  T={F['T']} R={F['R']}"
          f"  comm={COMM_STAGE_A:.0%} (actual {F.get('commission',COMMISSION):.0%})")
    print("#" * 72)

    race_list = [{**F, "commission": COMM_STAGE_A}]  # 0% commission Stage A override
    env_r     = DirectionalRaceEnv(race_list, seed=SEED + race_idx)

    # Random baseline
    rand_pnl = rollout(env_r, policy=None, n=20, seed=SEED)
    print(f"  random:     pnl ${rand_pnl.mean():.2f}  | do-nothing: $0.00")

    # Train: fresh model, fresh buffer
    model, venv = make_model(DirectionalRaceEnv(race_list, seed=SEED + race_idx),
                             seed=SEED + race_idx,
                             target_entropy=TARGET_ENTROPY_STAGE_A)
    cb = Metrics()

    model.learn(total_timesteps=PASSES_PER_RACE * F["T"], callback=cb, reset_num_timesteps=True)

    # Eval trained
    eval_env = DirectionalRaceEnv(race_list, seed=SEED + race_idx + 1000)
    trained_pnl = rollout(eval_env, policy=model, n=20, deterministic=True)
    gap = trained_pnl.mean() - 0.0   # vs do-nothing
    status = "PASS" if trained_pnl.mean() >= -1.0 else "FAIL"

    print(f"  trained: pnl ${trained_pnl.mean():.2f}"
          f" (vs do-nothing $0.00 -> gap {gap:+.2f}) | [{status}]")
    direction_audit(eval_env, model, n=10)

    # Stake-fraction distribution
    _stakes = []
    obs, _ = eval_env.reset(); done = False
    while not done:
        a, _ = model.predict(np.array([obs]), deterministic=True)
        sf = (float(a[0, MAX_RUNNERS]) + 1.0) / 2.0
        _stakes.append(sf)
        obs, _, done, _, _ = eval_env.step(a[0])
    print(f"  stake_frac: mean={np.mean(_stakes):.3f}  p10={np.percentile(_stakes,10):.3f}"
          f"  p90={np.percentile(_stakes,90):.3f}  near-zero(<0.05): "
          f"{int(100*np.mean(np.array(_stakes)<0.05))}%")

    STAGE_A_RESULTS.append(dict(
        race=name, random=rand_pnl.mean(), trained=trained_pnl.mean(),
        gap=gap, status=status))

    del model, venv, cb, env_r, eval_env

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 72)
print("  STAGE A SUMMARY")
print("=" * 72)
passes = sum(1 for r in STAGE_A_RESULTS if r["status"] == "PASS")
print(f"  Races tested : {len(STAGE_A_RESULTS)}")
print(f"  PASS         : {passes}/{len(STAGE_A_RESULTS)}")
for r in STAGE_A_RESULTS:
    print(f"  [{r['status']:4s}]  {r['race'][-30:]:30s}  "
          f"random ${r['random']:>8.2f}  trained ${r['trained']:>8.2f}  gap {r['gap']:>+8.2f}")
print()
if passes >= 1:
    print("  ✓ Stage A gate PASSED — proceed to Stage B (difficulty curriculum)")
else:
    print("  ✗ Stage A gate FAILED — diagnose before proceeding")
    print("    Checks:")
    print("    1. stake_frac near-zero% high? → agent found do-nothing (correct)")
    print("    2. stake_frac near-zero% low?  → entropy still collapsing, raise ENT_COEF_FLOOR")
    print("    3. direction wrong?            → check fill-price convention (bb vs bl)")
